<a href="https://colab.research.google.com/github/sam-wahid/vlm-llm-segmentation/blob/main/kd1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision torchaudio
!pip install -q open_clip_torch
!pip install -q transformers
!pip install -q pynvml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.3 MB/s eta 0:00:00


In [ ]:
import torch
import open_clip
import time
import pandas as pd

from torchvision.datasets import CIFAR10
from torchvision import transforms
from torch.utils.data import DataLoader

from transformers import AutoProcessor
from transformers import AutoModel

from pynvml import *

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

dataset = CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=False
)

classes = dataset.classes

print(classes)

100%|██████████| 170M/170M [00:02<00:00, 78.9MB/s]


['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


In [ ]:
# Load CLIP

import torch
import open_clip
import time

device = "cuda" if torch.cuda.is_available() else "cpu"

model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32',
    pretrained='laion2b_s34b_b79k'
)

tokenizer = open_clip.get_tokenizer('ViT-B-32')

model.to(device)
model.eval()

# Create prompts

prompts = [f"a photo of a {c}" for c in classes]

text = tokenizer(prompts).to(device)

with torch.no_grad():
    text_features = model.encode_text(text)

# Benchmark

correct = 0
total = 0

torch.cuda.empty_cache()

start_time = time.time()

for images, labels in loader:

    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():

        image_features = model.encode_image(images)

        logits = image_features @ text_features.T

        preds = logits.argmax(dim=1)

    correct += (preds == labels).sum().item()

    total += labels.size(0)

end_time = time.time()

# Metrics

clip_accuracy = 100 * correct / total

clip_total_time = end_time - start_time

clip_fps = total / clip_total_time

clip_latency = clip_total_time / total

clip_memory = (
    torch.cuda.memory_allocated() / 1024**2
)

print("===== CLIP RESULTS =====")

print("Accuracy:", clip_accuracy)
print("FPS:", clip_fps)
print("Latency:", clip_latency)
print("GPU Memory:", clip_memory, "MB")

===== CLIP RESULTS =====
Accuracy: 77.61
FPS: 6.049830403033083
Latency: 0.16529388980865478
GPU Memory: 0.0 MB


In [ ]:
from transformers import AutoProcessor
from transformers import AutoModel

import torch
import time

from torch.utils.data import Subset
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------------
# Small CIFAR-10 subset
# -----------------------------------

small_dataset = Subset(dataset, range(1000))

loader = DataLoader(
    small_dataset,
    batch_size=64,
    shuffle=False
)

# -----------------------------------
# Load SigLIP
# -----------------------------------

siglip_model = AutoModel.from_pretrained(
    "google/siglip-base-patch16-224"
).to(device)

processor = AutoProcessor.from_pretrained(
    "google/siglip-base-patch16-224"
)

siglip_model.eval()

# -----------------------------------
# Text prompts
# -----------------------------------

prompts = [f"a photo of a {c}" for c in classes]

text_inputs = processor(
    text=prompts,
    padding=True,
    return_tensors="pt"
).to(device)

# -----------------------------------
# Text embeddings
# -----------------------------------

with torch.no_grad():

    text_outputs = siglip_model.text_model(
        **text_inputs
    )

    text_features = text_outputs.pooler_output

# -----------------------------------
# Benchmark
# -----------------------------------

correct = 0
total = 0

torch.cuda.empty_cache()

start_time = time.time()

for images, labels in loader:

    images = images.to(device)
    labels = labels.to(device)

    with torch.no_grad():

        image_outputs = siglip_model.vision_model(
            pixel_values=images
        )

        image_features = image_outputs.pooler_output

        # Similarity

        logits = image_features @ text_features.T

        preds = logits.argmax(dim=1)

    correct += (preds == labels).sum().item()

    total += labels.size(0)

end_time = time.time()

# -----------------------------------
# Metrics
# -----------------------------------

siglip_accuracy = 100 * correct / total

siglip_total_time = end_time - start_time

siglip_fps = total / siglip_total_time

siglip_latency = siglip_total_time / total

siglip_memory = (
    torch.cuda.max_memory_allocated() / 1024**2
)

# -----------------------------------
# Results
# -----------------------------------

print("===== SigLIP RESULTS =====")

print("Accuracy:", siglip_accuracy)

print("FPS:", siglip_fps)

print("Latency:", siglip_latency)

print("GPU Memory:", siglip_memory, "MB")

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

===== SigLIP RESULTS =====
Accuracy: 7.8
FPS: 1.216798309740503
Latency: 0.8218288865089417
GPU Memory: 0.0 MB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
